In [8]:
!nvidia-smi
!nvcc --version | tail -2
!python --version

Wed Oct  7 19:43:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             27W /   70W |     109MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!find /content/drive/MyDrive -maxdepth 3 \( -name "*dynamo*" -o -name "*int8_static_matmul*" -o -name "calib_tiles.npy" \) -exec ls -la {} \;

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root 367525888 Oct  5 10:06 /content/drive/MyDrive/edge_results/models/sam_vitb_encoder_fp32_dynamo.onnx.data
-rw------- 1 root root 788681 Oct  5 10:06 /content/drive/MyDrive/edge_results/models/sam_vitb_encoder_fp32_dynamo.onnx
-rw------- 1 root root 109881386 Oct  5 20:01 /content/drive/MyDrive/edge_results/models/sam_vitb_encoder_int8_static_matmul.onnx
-rw------- 1 root root 717226112 Oct  5 15:28 /content/drive/MyDrive/edge_results/calibration/calib_tiles.npy
total 345398
drwx------  2 root root      4096 Oct  7 14:23 .
drwx------ 13 root root      4096 Oct  7 14:23 ..
-rw-------  1 root root 169411710 Oct  5 11:06 Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100_masks.tif
-rw-------  1 root root 184267258 Oct  5 11:43 PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.ome_deconv_masks.tif
total 345398
drwx------  2 root root      4096 Oct  7 14:23

In [ ]:
# Cell 3
!pip install -q tensorrt polygraphy onnx onnxruntime-gpu colored
import tensorrt as trt
print("TensorRT", trt.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 5.0 MB/s eta 0:00:00
TensorRT 11.3.0.99


In [ ]:
!which trtexec
!find / -name trtexec -type f 2>/dev/null | head


In [ ]:
import tensorrt as trt, glob, shutil, subprocess
print("TensorRT", trt.__version__)

print("=== trtexec ===")
print(shutil.which("trtexec"))
print(subprocess.run("find / -name 'trtexec*' -type f 2>/dev/null | head", shell=True, capture_output=True, text=True).stdout)

print("=== BuilderFlag members ===")
print([f for f in dir(trt.BuilderFlag) if f.isupper()])

print("=== NetworkDefinitionCreationFlag members ===")
print([f for f in dir(trt.NetworkDefinitionCreationFlag) if f.isupper()])

print("=== can the builder see the T4? ===")
logger = trt.Logger(trt.Logger.INFO)
b = trt.Builder(logger)
config = b.create_builder_config()
# In TensorRT 10+, we can check platform support for precision types via config flags
print("platform has fast fp16 (via config flag check):", config.get_flag(trt.BuilderFlag.FP16) if hasattr(trt.BuilderFlag, 'FP16') else "N/A")

print("=== installed trt packages ===")
!pip list 2>/dev/null | grep -i -E "tensorrt|polygraphy|onnx|modelopt"

TensorRT 11.3.0.99
=== trtexec ===
None

=== BuilderFlag members ===
['DEBUG', 'DIRECT_IO', 'DISABLE_COMPILATION_CACHE', 'DISABLE_TIMING_CACHE', 'DISTRIBUTIVE_INDEPENDENCE', 'EDITABLE_TIMING_CACHE', 'ERROR_ON_TIMING_CACHE_MISS', 'EXCLUDE_LEAN_RUNTIME', 'GPU_FALLBACK', 'MONITOR_MEMORY', 'REFIT', 'REFIT_IDENTICAL', 'REFIT_INDIVIDUAL', 'SAFETY_SCOPE', 'SPARSE_WEIGHTS', 'STRICT_NANS', 'STRIP_PLAN', 'TF32', 'VERSION_COMPATIBLE', 'WEIGHT_STREAMING']
=== NetworkDefinitionCreationFlag members ===
['PREFER_AOT_PYTHON_PLUGINS', 'PREFER_JIT_PYTHON_PLUGINS', 'STRONGLY_TYPED']
=== can the builder see the T4? ===
platform has fast fp16 (via config flag check): N/A
=== installed trt packages ===
onnx                                  1.23.2
onnxruntime-gpu                       1.30.0
polygraphy                            0.53.6
tensorrt                              11.3.0.99
tensorrt_cu13                         11.3.0.99
tensorrt_cu13_bindings                11.3.0.99
tensorrt_cu13_libs             

In [6]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/work/models
!cp /content/drive/MyDrive/edge_results/models/sam_vitb_encoder_fp32_dynamo.onnx* /content/work/models/
!cp /content/drive/MyDrive/edge_results/models/sam_vitb_encoder_int8_static_matmul.onnx /content/work/models/
!ls -la /content/work/models


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 467008
drwxr-xr-x 2 root root      4096 Oct  7 19:43 .
drwxr-xr-x 3 root root      4096 Oct  7 19:43 ..
-rw------- 1 root root    788681 Oct  7 19:43 sam_vitb_encoder_fp32_dynamo.onnx
-rw------- 1 root root 367525888 Oct  7 19:43 sam_vitb_encoder_fp32_dynamo.onnx.data
-rw------- 1 root root 109881386 Oct  7 19:43 sam_vitb_encoder_int8_static_matmul.onnx


In [7]:
import tensorrt as trt, numpy as np, onnx, os
from onnx import helper, TensorProto

# 1) T4 test: tiny FP16 ReLU model, built strongly typed
x = helper.make_tensor_value_info("x", TensorProto.FLOAT16, [1, 3, 8, 8])
y = helper.make_tensor_value_info("y", TensorProto.FLOAT16, [1, 3, 8, 8])
g = helper.make_graph([helper.make_node("Relu", ["x"], ["y"])], "tiny", [x], [y])
m = helper.make_model(g, opset_imports=[helper.make_opsetid("", 17)])
onnx.save(m, "/content/work/tiny.onnx")

logger = trt.Logger(trt.Logger.WARNING)
b = trt.Builder(logger)
net = b.create_network()
p = trt.OnnxParser(net, logger)
print("parsed:", p.parse(open("/content/work/tiny.onnx", "rb").read()))
plan = b.build_serialized_network(net, b.create_builder_config())
print("TINY ENGINE BUILT ON T4:", plan is not None)

# 2) ModelOpt install and FP16 conversion
!pip install -q --extra-index-url https://pypi.nvidia.com "nvidia-modelopt[onnx]" 2>&1 | tail -5
import modelopt
print("modelopt", modelopt.__version__)
import modelopt.onnx.autocast as autocast
src = "/content/work/models/sam_vitb_encoder_fp32_dynamo.onnx"
dst = "/content/work/models/sam_vitb_encoder_fp16_autocast.onnx"
mm = autocast.convert_to_mixed_precision(onnx_path=src, low_precision_type="fp16", keep_io_types=True)
onnx.save(mm, dst)
print("fp16 model MB:", os.path.getsize(dst) / 1e6)



parsed: True
TINY ENGINE BUILT ON T4: True


ModuleNotFoundError: No module named 'modelopt'

In [9]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

os.makedirs("/content/work/models", exist_ok=True)
D = "/content/drive/MyDrive/edge_results/models"
for f in ["sam_vitb_encoder_fp32_dynamo.onnx", "sam_vitb_encoder_fp32_dynamo.onnx.data",
          "sam_vitb_encoder_int8_static_matmul.onnx"]:
    if not os.path.exists(f"/content/work/models/{f}"):
        subprocess.run(["cp", f"{D}/{f}", "/content/work/models/"], check=True)
print(subprocess.run("ls -la /content/work/models", shell=True, capture_output=True, text=True).stdout)

!pip install -q onnx onnxconverter-common 2>&1 | tail -3
import onnx
from onnxconverter_common import float16

src = "/content/work/models/sam_vitb_encoder_fp32_dynamo.onnx"
dst = "/content/work/models/sam_vitb_encoder_fp16.onnx"
m = onnx.load(src)                       # loads the .onnx.data weights too
m16 = float16.convert_float_to_float16(m, keep_io_types=True)
onnx.save(m16, dst)
print("fp16 model MB:", os.path.getsize(dst) / 1e6)
print("nodes:", len(m16.graph.node), "| ops:", sorted({n.op_type for n in m16.graph.node}))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 467008
drwxr-xr-x 2 root root      4096 Oct  7 19:43 .
drwxr-xr-x 3 root root      4096 Oct  7 19:43 ..
-rw------- 1 root root    788681 Oct  7 19:43 sam_vitb_encoder_fp32_dynamo.onnx
-rw------- 1 root root 367525888 Oct  7 19:43 sam_vitb_encoder_fp32_dynamo.onnx.data
-rw------- 1 root root 109881386 Oct  7 19:43 sam_vitb_encoder_int8_static_matmul.onnx



/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 3.6217759902257285e-09 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -9.33175314798973e-08 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 3.306911722233963e-08 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -8.085468294893872e-09 will be truncated to -1e-07
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 9.46894473941029e-09 will be truncated to 1e-07
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -3.858410124735201e

fp16 model MB: 184.526874
nodes: 566 | ops: ['Add', 'Cast', 'Conv', 'Div', 'Einsum', 'Gelu', 'LayerNormalization', 'MatMul', 'Mul', 'Pad', 'Pow', 'ReduceMean', 'Reshape', 'Slice', 'Softmax', 'Sqrt', 'Squeeze', 'Sub', 'Transpose', 'Unsqueeze']


In [10]:
print("done")

done


In [11]:
import tensorrt as trt, torch, json, time, os, subprocess, numpy as np
M = "/content/work/models"; E = "/content/work/engines"; os.makedirs(E, exist_ok=True)
R = "/content/drive/MyDrive/edge_results/results_colab"; os.makedirs(R, exist_ok=True)
logger = trt.Logger(trt.Logger.WARNING)

def clocks():
    return subprocess.run("nvidia-smi --query-gpu=clocks.sm,clocks.max.sm,temperature.gpu,power.draw --format=csv,noheader",
                          shell=True, capture_output=True, text=True).stdout.strip()

def build(onnx_path, plan_path):
    b = trt.Builder(logger)
    net = b.create_network()                      # strongly typed by default in TensorRT 11
    p = trt.OnnxParser(net, logger)
    if not p.parse_from_file(onnx_path):          # parse_from_file finds the .onnx.data next to it
        for i in range(p.num_errors): print(p.get_error(i))
        raise SystemExit("parse failed")
    cfg = b.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)
    cfg.profiling_verbosity = trt.ProfilingVerbosity.DETAILED
    t = time.time()
    plan = b.build_serialized_network(net, cfg)
    if plan is None: raise SystemExit("build failed")
    open(plan_path, "wb").write(plan)
    return round(time.time() - t, 1)

TORCH = {trt.float32: torch.float32, trt.float16: torch.float16}

def bench(plan_path, runs=30, warm=10):
    eng = trt.Runtime(logger).deserialize_cuda_engine(open(plan_path, "rb").read())
    ctx = eng.create_execution_context()
    names = [eng.get_tensor_name(i) for i in range(eng.num_io_tensors)]
    g = torch.Generator().manual_seed(0); bufs = {}; out = None
    for n in names:
        shp = tuple(eng.get_tensor_shape(n)); dt = TORCH[eng.get_tensor_dtype(n)]
        if eng.get_tensor_mode(n) == trt.TensorIOMode.INPUT:
            bufs[n] = torch.randn(shp, generator=g).to(dt).cuda()
        else:
            bufs[n] = torch.empty(shp, dtype=dt, device="cuda"); out = n
        ctx.set_tensor_address(n, bufs[n].data_ptr())
    s = torch.cuda.Stream(); ms = []
    for i in range(warm + runs):
        a, z = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        a.record(s); ctx.execute_async_v3(s.cuda_stream); z.record(s); z.synchronize()
        if i >= warm: ms.append(a.elapsed_time(z))
    ms = np.array(ms)
    info = json.loads(eng.create_engine_inspector().get_engine_information(trt.LayerInformationFormat.JSON))
    layers = info.get("Layers", [])
    half = sum("Half" in json.dumps(l) for l in layers)
    return dict(median_ms=round(float(np.median(ms)), 2), p10_ms=round(float(np.percentile(ms, 10)), 2),
                p90_ms=round(float(np.percentile(ms, 90)), 2), runs=runs, warmup=warm,
                plan_MB=round(os.path.getsize(plan_path) / 1e6, 1), n_layers=len(layers), layers_with_half=half,
                out_dtype=str(bufs[out].dtype)), bufs[out].float().cpu().flatten()

results, outs = {}, {}
for name, f in [("fp32", "sam_vitb_encoder_fp32_dynamo.onnx"), ("fp16", "sam_vitb_encoder_fp16.onnx")]:
    path = f"{M}/{f}"
    assert os.path.exists(path), f"missing {path}: rerun the FP16 conversion cell"
    print(f"--- {name}: ONNX {os.path.getsize(path)/1e6:.1f} MB | clocks before: {clocks()}")
    t = build(path, f"{E}/{name}.plan")
    r, o = bench(f"{E}/{name}.plan"); r["build_s"] = t; r["clocks_after"] = clocks()
    results[name], outs[name] = r, o
    print(name, r)

cos = torch.nn.functional.cosine_similarity(outs["fp32"], outs["fp16"], dim=0).item()
results["sanity_cosine_fp16_vs_fp32_random_input"] = cos
results["versions"] = dict(tensorrt=trt.__version__, torch=torch.__version__, gpu=torch.cuda.get_device_name(0))
json.dump(results, open(f"{R}/stage4_trt_fp32_fp16.json", "w"), indent=2)
print("cosine FP16 vs FP32 (random input, sanity only):", cos)


--- fp32: ONNX 0.8 MB | clocks before: 585 MHz, 1590 MHz, 56, 28.25 W
fp32 {'median_ms': 314.45, 'p10_ms': 308.86, 'p90_ms': 318.94, 'runs': 30, 'warmup': 10, 'plan_MB': 370.3, 'n_layers': 203, 'layers_with_half': 0, 'out_dtype': 'torch.float32', 'build_s': 29.3, 'clocks_after': '1500 MHz, 1590 MHz, 65, 40.64 W'}
--- fp16: ONNX 184.5 MB | clocks before: 1500 MHz, 1590 MHz, 65, 40.64 W
fp16 {'median_ms': 108.26, 'p10_ms': 106.69, 'p90_ms': 109.29, 'runs': 30, 'warmup': 10, 'plan_MB': 199.5, 'n_layers': 228, 'layers_with_half': 226, 'out_dtype': 'torch.float32', 'build_s': 41.8, 'clocks_after': '1575 MHz, 1590 MHz, 64, 36.66 W'}
cosine FP16 vs FP32 (random input, sanity only): 1.000023365020752


In [14]:
import shutil
src = "/content/drive/MyDrive/edge_results/models/sam_vitb_encoder_int8_trt_matmul.onnx"
path = f"{M}/sam_vitb_encoder_int8_trt_matmul.onnx"
if not os.path.exists(path): shutil.copy(src, path)
print("ONNX MB:", round(os.path.getsize(path) / 1e6, 1), "| clocks before:", clocks())

t = build(path, f"{E}/int8_trt.plan")
r, o = bench(f"{E}/int8_trt.plan")
r.update(build_s=t, clocks_after=clocks(), layers_with_int8=count(f"{E}/int8_trt.plan", "Int8"))
print("int8_trt", r)
cos = torch.nn.functional.cosine_similarity(outs["fp32"], o, dim=0).item()
print("cosine INT8 vs FP32 (random input, sanity only):", cos)
results["int8_trt"], results["sanity_cosine_int8_trt_vs_fp32_random_input"] = r, cos
json.dump(results, open(f"{R}/stage4_trt_all.json", "w"), indent=2)

ONNX MB: 109.2 | clocks before: 585 MHz, 1590 MHz, 55, 27.95 W
int8_trt {'median_ms': 98.74, 'p10_ms': 98.18, 'p90_ms': 99.46, 'runs': 30, 'warmup': 10, 'plan_MB': 111.9, 'n_layers': 226, 'layers_with_half': 0, 'out_dtype': 'torch.float32', 'build_s': 12.7, 'clocks_after': '1560 MHz, 1590 MHz, 60, 31.87 W', 'layers_with_int8': 194}
cosine INT8 vs FP32 (random input, sanity only): 0.9856246709823608


In [15]:
import shutil
src = "/content/trt_check_tiles.npz"
d = np.load(src); imgs, ref = d["images"], d["reference"]
print("tiles", imgs.shape, "reference", ref.shape)

def encode(plan_path, x_np):
    eng = trt.Runtime(logger).deserialize_cuda_engine(open(plan_path, "rb").read())
    ctx = eng.create_execution_context(); bufs = {}; out = None
    for n in [eng.get_tensor_name(i) for i in range(eng.num_io_tensors)]:
        shp = tuple(eng.get_tensor_shape(n)); dt = TORCH[eng.get_tensor_dtype(n)]
        if eng.get_tensor_mode(n) == trt.TensorIOMode.INPUT:
            bufs[n] = torch.zeros(shp, dtype=dt, device="cuda"); inp = n
        else:
            bufs[n] = torch.empty(shp, dtype=dt, device="cuda"); out = n
        ctx.set_tensor_address(n, bufs[n].data_ptr())
    s = torch.cuda.Stream(); res = []
    for x in x_np:
        bufs[inp].copy_(torch.from_numpy(x[None]).to(bufs[inp].dtype))
        ctx.execute_async_v3(s.cuda_stream); s.synchronize()
        res.append(bufs[out].float().cpu().numpy()[0])
    return np.stack(res)

def compare(a, b):
    a, b = a.astype(np.float64).ravel(), b.astype(np.float64).ravel()
    return dict(cosine=float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))),
                max_abs_diff=float(np.abs(a - b).max()), mean_abs_diff=float(np.abs(a - b).mean()))

check = {}
for name in ["fp32", "fp16", "int8_trt"]:
    out = encode(f"{E}/{name}.plan", imgs)
    tiles = [compare(ref[i], out[i]) for i in range(len(ref))]
    check[name] = dict(min_cosine=min(t["cosine"] for t in tiles),
                       max_abs_diff=max(t["max_abs_diff"] for t in tiles), tiles=tiles)
    print(f"{name:9s} min cosine {check[name]['min_cosine']:.5f} | worst max|diff| {check[name]['max_abs_diff']:.3f} | "
          f"per-tile cosines {[round(t['cosine'], 4) for t in tiles]}")
json.dump(check, open(f"{R}/stage4_trt_real_tile_check.json", "w"), indent=2)


tiles (5, 3, 1024, 1024) reference (5, 256, 64, 64)
fp32      min cosine 1.00000 | worst max|diff| 0.000 | per-tile cosines [1.0, 1.0, 1.0, 1.0, 1.0]
fp16      min cosine 0.99998 | worst max|diff| 0.035 | per-tile cosines [1.0, 1.0, 1.0, 1.0, 1.0]
int8_trt  min cosine 0.98447 | worst max|diff| 0.401 | per-tile cosines [0.9845, 0.993, 0.9927, 0.9935, 0.9915]
